In [1]:
from attention import ATTENTION_IMPLEMENTATIONS, MultiHeadAttention
import torch
import time
from itertools import product
import pandas as pd

In [2]:
device = "cuda"
n_iters = 100
warmup_iters = 10
embedding_dims = 768
num_heads = 12
attn_dims = int(embedding_dims / num_heads)

# Simple timing of matmul vs einsum

In [3]:
batch_sizes = [1, 2, 4]
seq_lengths = [128, 512, 1024, 2048]

## Forward pass only

In [4]:
attention = MultiHeadAttention(
    embedding_dims, attn_dims, num_heads, attn_impl=ATTENTION_IMPLEMENTATIONS.MATMUL
).to(device)

In [5]:
matmul_times = []
einsum_times = []
batch_size_list = []
seq_length_list = []

In [6]:
def get_total_time(X):
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    total_time = 0
    with torch.no_grad():
        for x in X[:warmup_iters]:
            attention(x)

        for x in X[warmup_iters:]:
            start_event.record()
            attention(x)
            end_event.record()
            torch.cuda.synchronize()
            total_time += start_event.elapsed_time(end_event)
    return total_time

In [7]:
for batch_size, seq_length in product(batch_sizes, seq_lengths):
    batch_size_list.append(batch_size)
    seq_length_list.append(seq_length)

    X = torch.randn((n_iters, batch_size, seq_length, embedding_dims)).to(device)
    
    attention.attn_impl = ATTENTION_IMPLEMENTATIONS.MATMUL
    matmul_times.append(get_total_time(X))

    attention.attn_impl = ATTENTION_IMPLEMENTATIONS.EINSUM
    einsum_times.append(get_total_time(X))

In [8]:
df = pd.DataFrame({
        "batch size": batch_size_list, 
        "sequence length": seq_length_list, 
        "matmul_times": matmul_times,
        "einsum_times": einsum_times
    })
df

,batch size,sequence length,matmul_times,einsum_times
0,1,128,10.494720,12.232736
1,1,512,33.314144,32.696256
2,1,1024,103.995552,103.232960
3,1,2048,357.886944,363.076734
4,2,128,14.024800,14.728160
5,2,512,63.606176,64.199360
6,2,1024,215.036928,215.896543
7,2,2048,738.537277,739.653149
8,4,128,21.298976,22.763392
9,4,512,133.457312,133.851423


## Forward and backward 

In [4]:
attention = MultiHeadAttention(
    embedding_dims, attn_dims, num_heads, attn_impl=ATTENTION_IMPLEMENTATIONS.MATMUL
).to(device)

ce_loss = torch.nn.CrossEntropyLoss().to(device)

In [5]:
matmul_times = []
einsum_times = []
batch_size_list = []
seq_length_list = []

In [6]:
def get_total_time(X, Y):
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    total_time = 0
    for x, y in zip(X[:warmup_iters], Y[:warmup_iters]):
        attention.zero_grad()
        x_hat = attention(x)
        loss = ce_loss(x, y)
        loss.requires_grad = True
        loss.backward()

    for x, y in zip(X[warmup_iters:], Y[warmup_iters:]):
        start_event.record()
        attention.zero_grad()
        x_hat = attention(x)
        loss = ce_loss(x, y)
        loss.requires_grad= True
        loss.backward()
        end_event.record()
        torch.cuda.synchronize()
        total_time += start_event.elapsed_time(end_event) 
    return total_time

In [7]:
for batch_size, seq_length in product(batch_sizes, seq_lengths):
    X = torch.randn((n_iters, batch_size, seq_length, embedding_dims)).to(device)
    Y = torch.randn((n_iters, batch_size, seq_length, embedding_dims)).to(device)

    batch_size_list.append(batch_size)
    seq_length_list.append(seq_length)
    
    attention.attn_impl = ATTENTION_IMPLEMENTATIONS.MATMUL
    matmul_times.append(get_total_time(X, Y))

    attention.attn_impl = ATTENTION_IMPLEMENTATIONS.EINSUM
    einsum_times.append(get_total_time(X, Y))

In [8]:
df = pd.DataFrame({
        "batch size": batch_size_list, 
        "sequence length": seq_length_list, 
        "matmul_times": matmul_times,
        "einsum_times": einsum_times
    })
df

,batch size,sequence length,matmul_times,einsum_times
0,1,128,21.096416,23.376352
1,1,512,62.154144,57.401920
2,1,1024,151.883007,149.905824
3,1,2048,494.892832,491.812065
4,2,128,23.034880,25.372576
5,2,512,91.279776,93.922529
6,2,1024,264.096961,264.557822
7,2,2048,867.664092,812.708292
8,4,128,29.529344,29.385568
9,4,512,159.267968,159.294784
